# webSLM — fine-tune a domain base model (clone & run)

Fine-tunes a small Qwen base on your **medical / legal / insurance** (or any) data with
**LoRA**, merges it, and **pushes a full model to Hugging Face**. The webSLM build pipeline
then quantizes that HF model into a WebGPU `.wasm` for the browser.

```
this notebook (train + merge + push)  ->  your model on HF  ->  webSLM build-slm.yml  ->  .wasm
```

**Before you start:** Runtime -> Change runtime type -> **T4 GPU** (free). A 0.5B base trains
comfortably on a T4. This notebook does NOT need MLC/emscripten — quantization happens later
in the webSLM repo's GitHub Action.

## 1. Check the GPU

In [ ]:
!nvidia-smi -L || print('No GPU! Runtime -> Change runtime type -> T4 GPU')

## 2. Clone webSLM + install deps

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/vishalmysore/webSLM.git
%cd /content/webSLM
!pip install -q -r finetune/requirements.txt

## 3. (Optional) Use YOUR OWN data
The repo ships tiny starter sets at `finetune/data/{medical,legal,insurance}.jsonl` — enough to
smoke-test the pipeline, **not** enough to truly specialize a model. Replace/extend them with
your real data (same chat format, one JSON object per line):
```json
{"messages":[{"role":"system","content":"..."},{"role":"user","content":"..."},{"role":"assistant","content":"..."}]}
```
Upload your file via the Colab Files panel (drag it into `/content/webSLM/finetune/data/`) and
point `DATA` at it in the next cell.

In [ ]:
# Peek at the starter data (replace with your own for a real fine-tune)
!head -n 2 finetune/data/medical.jsonl

## 4. Log in to Hugging Face
Get a **Write** token at https://huggingface.co/settings/tokens, run this cell, and paste it.
The merged model repo is created automatically on push.

In [ ]:
from huggingface_hub import login
login()  # paste your HF write token

## 5. Train -> merge -> push
Edit the variables, then run. `HF_REPO` is where the finished model lands (use your namespace).

In [ ]:
import os
# ---- EDIT THESE ----
DOMAIN  = 'medical'                       # medical | legal | insurance | (your own file's name)
BASE    = 'Qwen/Qwen2.5-0.5B-Instruct'    # small Instruct base (ungated, browser-friendly)
HF_USER = 'yourname'                      # your Hugging Face username/namespace
EPOCHS  = 3
# --------------------
DATA    = f'finetune/data/{DOMAIN}.jsonl' # or your uploaded file
HF_REPO = f'{HF_USER}/WebSLM-{DOMAIN.capitalize()}-0.5B'
os.environ.update(DOMAIN=DOMAIN, BASE=BASE, DATA=DATA, HF_REPO=HF_REPO, EPOCHS=str(EPOCHS))
print('Training', BASE, 'on', DATA, '-> pushing to', HF_REPO)

In [ ]:
!python finetune/train_lora.py \
    --base "$BASE" \
    --data "$DATA" \
    --push-merged "$HF_REPO" \
    --epochs "$EPOCHS"

## 6. Build it into a browser SLM
Your merged model is now on HF. Go to the **webSLM** repo -> **Actions** ->
**"Build domain SLM (WebGPU/WebLLM)"** -> *Run workflow*:
- **Domain preset** = `Custom`
- **custom_model_hf** = your `HF_REPO` (e.g. `yourname/WebSLM-Medical-0.5B`)
- **custom_arch** = `qwen2`,  **custom_conv** = `qwen2`
- **custom_name** = `WebSLM-Medical-0.5B`
- tick **upload_hf** if you set `HF_TOKEN` + `HF_NAMESPACE` on the repo

~45 min later you get the `.wasm` (Release + artifact) — load it in `demo/index.html`.

---
**Reality check:** a dozen starter examples won't meaningfully specialize a model — they prove
the wiring. For real domain quality, bring hundreds-to-thousands of high-quality examples, and
always keep a human in the loop for medical/legal/insurance outputs.